<a href="https://colab.research.google.com/github/taipinshe/test/blob/main/HC3%E6%88%90%E5%8A%9F%E6%9C%89%E6%95%88%E7%9A%84%E7%A8%8B%E5%BC%8F%E7%A2%BC_%E5%BB%B6%E7%BA%8CV5_2_%E5%81%B5%E6%B8%AC%E7%B9%81%E9%AB%94%E4%B8%AD%E6%96%87%E7%89%88_V1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# 儲存格 1：載入 HC3 官方微調中文 AI 偵測模型與特徵引擎 (只需執行一次)
# ==============================================================================
!pip install -q "transformers>=4.40.0" "datasets>=2.18.0" "accelerate>=0.28.0" scikit-learn seaborn matplotlib python-docx pymupdf

import os
import re
import json
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import HTML, display
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 載入 HC3 官方已在大量中文數據上微調完畢的 AI 偵測模型權重 (非隨機初始化)
DETECTOR_MODEL = "Hello-SimpleAI/chatgpt-detector-roberta-chinese"

print(f"運行裝置: {DEVICE}")
print("正在從 HuggingFace 下載預訓練中文 AI 偵測模型權重...")

tokenizer = AutoTokenizer.from_pretrained(DETECTOR_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(DETECTOR_MODEL).to(DEVICE)
model.eval()

# ------------------------------------------------------------------------------
# 繁體中文學術 AI 潤稿特徵庫
# ------------------------------------------------------------------------------
AI_CHINESE_CLICHES = {
    "值得注意的是", "不可否認的是", "毋庸置疑", "綜上所述", "總結來說", "總體而言",
    "換言之", "顯而易見", "誠然", "不難發現", "有鑑於此", "在此背景下", "在當今",
    "至關重要", "舉足輕重", "不可忽視", "不容忽視", "不容小覷", "扮演著關鍵角色",
    "奠定堅實基礎", "重中之重", "不可或缺", "不言而喻", "深遠的影響", "深遠影響",
    "雙刃劍", "基石", "催化劑", "橋樑", "藍圖", "多維度", "全方位", "賦能", "跨越式",
    "新範式", "範式轉移", "織就", "紐帶", "燈塔", "引領者", "新紀元", "新篇章",
    "深入探討", "全面解析", "深入剖析", "彰顯了", "凸顯出", "深刻體現", "旨在探討",
    "有助於進一步", "為未來的發展", "發揮著積極作用", "提供有力支撐", "推動了", "邁出了堅實的一步"
}

AI_PATTERNS = [
    r"不僅(?:能|可以|在於).*?更(?:能|可以|在於)",
    r"既(?:能|要|是).*?又(?:能|要|是)",
    r"隨著.*?的(?:飛速|快速|日益).*?,",
    r"在.*?的浪潮下",
]

def is_noise_or_heading(text):
    """過濾目錄、標題、頁碼、表格編號等非論述性文字"""
    s = text.strip()
    # 目錄/標題模式 (如: 第一節...、表1...、圖2...、頁碼...)
    if re.match(r'^(?:第[一二三四五六七八九十0-9]+[章節]|表\s*\d+|圖\s*\d+|附錄[一二三四0-9]+|\d+\.\d+|\([一二三四0-9]+\))\s*', s) and len(s) < 30:
        return True
    if re.match(r'^(?:關鍵詞|關鍵字|指導教授|研究生|學系|學院|大學|目錄|摘要|References|Bibliography)[:：\t\s]?', s) and len(s) < 25:
        return True
    # 含有結尾 tab 頁碼 (如: ... \t12)
    if re.search(r'\t\s*\d+\s*$', s) and len(s) < 35:
        return True
    # 純數字或過短
    if len(re.findall(r'[\u4e00-\u9fff]', s)) < 8:
        return True
    return False

print("✅ 預訓練中文 AI 偵測引擎就緒！請執行【儲存格 2】。")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 62.1 MB/s eta 0:00:00
運行裝置: cuda
正在從 HuggingFace 下載預訓練中文 AI 偵測模型權重...


config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  409MB            

✅ 預訓練中文 AI 偵測引擎就緒！請執行【儲存格 2】。


model.safetensors: downloading bytes:           |  0.00B            

In [5]:
# ==============================================================================
# 儲存格 2：繁中論文上傳、嚴謹文獻過濾 (含中文/英文變體) 與診斷儀表板 (可重複執行)
# ==============================================================================
import os
import re
import json
import docx
import pymupdf
import torch
import numpy as np
import torch.nn.functional as F
from IPython.display import HTML, display
from google.colab import files

def extract_text_from_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ""
    try:
        if ext == ".docx":
            doc = docx.Document(file_path)
            elements = []
            for p in doc.paragraphs:
                if p.text.strip():
                    elements.append(p.text.strip())
            for table in doc.tables:
                for row in table.rows:
                    for cell in row.cells:
                        if cell.text.strip() and cell.text.strip() not in elements:
                            elements.append(cell.text.strip())
            text = "\n\n".join(elements)
        elif ext == ".pdf":
            with pymupdf.open(file_path) as doc:
                text = "\n\n".join([page.get_text() for page in doc])
        elif ext == ".txt":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
    except Exception as e:
        print(f"讀取失敗 {file_path}: {e}")
    return text

def clean_chinese_references_strict(text):
    """
    強化版繁中文獻過濾器：
    1. 支援「參考文獻、引用文獻、參考書目、中文參考文獻、西文參考文獻」
    2. 支援英文「Reference、References、Bibliography、Works Cited」
    3. 支援阿拉伯數字、羅馬數字、國字章節編號 (如: 7. 參考文獻 / 柒、參考文獻)
    """
    doc_len = len(text)
    if doc_len == 0:
        return text

    patterns = [
        r'(?i)(?:^|\n)\s*(?:(?:[0-9]{1,2}|[ivxlcdm]+|[一二三四五六七八九十壹貳參肆伍陸柒捌玖拾]+)[\.、．\s]*)?(?:(?:中|西|英|外)?文?參考(?:文獻|書目|資料)|引用(?:文獻|書目)|references?|bibliography|works\s+cited)(?:\s*(?:與|及|and)\s*附錄)?\s*[:：\r\n]?'
    ]

    cutoff_pos = doc_len
    for pat in patterns:
        for m in re.finditer(pat, text):
            # 確保是在文章中後半段 (超過 35% 處) 避免誤切內文
            if m.start() > doc_len * 0.35:
                cutoff_pos = min(cutoff_pos, m.start())
                break
        if cutoff_pos < doc_len:
            break

    clean_text = text[:cutoff_pos]
    if cutoff_pos < doc_len:
        print(f"✂️ 已成功過濾文獻區（原總長度: {doc_len} 字元 ➜ 裁切後正文: {cutoff_pos} 字元）")
    else:
        print("ℹ️ 未偵測到獨立「參考文獻 / References」標題，進行全文掃描。")
    return clean_text

def split_chinese_sentences(text):
    clean_text = clean_chinese_references_strict(text)
    raw_splits = re.split(r'([。！？；]+|\n{2,})', clean_text)
    valid_sentences = []

    for i in range(0, len(raw_splits)-1, 2):
        s = (raw_splits[i] + raw_splits[i+1]).strip()
        # 排除目錄、標題、頁碼、表格等非論述語句
        if not is_noise_or_heading(s):
            valid_sentences.append(s)

    if len(raw_splits) % 2 == 1:
        s_last = raw_splits[-1].strip()
        if not is_noise_or_heading(s_last):
            valid_sentences.append(s_last)

    return valid_sentences, clean_text

def analyze_chinese_document_real(text, model, tokenizer):
    if not text or len(text.strip()) < 30:
        return None

    sentences, clean_text = split_chinese_sentences(text)
    if not sentences:
        return None

    total_chinese_chars = len(re.findall(r'[\u4e00-\u9fff]', clean_text))
    matched_cliches_all = [c for c in AI_CHINESE_CLICHES if c in clean_text]

    model.eval()
    all_sentences = []

    for idx, s in enumerate(sentences):
        hit_words = [c for c in AI_CHINESE_CLICHES if c in s]
        hit_pats = sum(1 for p in AI_PATTERNS if re.search(p, s))

        s_inputs = tokenizer(s, truncation=True, max_length=512, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model(**s_inputs)
            probs = F.softmax(outputs.logits, dim=-1)[0]
            raw_model_ai_prob = probs[1].item() # index 1 为 ChatGPT 機率

        # 特徵加權校準
        cliche_boost = min(len(hit_words) * 0.18 + hit_pats * 0.15, 0.40)
        calibrated_score = raw_model_ai_prob * 0.70 + cliche_boost
        calibrated_score = min(max(calibrated_score, 0.01), 0.99)

        if calibrated_score >= 0.50 or (len(hit_words) >= 2 and calibrated_score >= 0.35):
            tier = "HIGH"
            reason = "經 HC3 模型判定具極高神經網絡 AI 規律，或密集出現公式化潤稿套詞。"
            advice = "建議將整句打散，刪除「不可或缺、賦能」等抽象詞，以第一人稱主動語態與具體數據重述。"
        elif calibrated_score >= 0.20 or len(hit_words) >= 1 or hit_pats >= 1:
            tier = "MED"
            reason = "文法完整，但包含特定 ChatGPT 典型學術轉折詞彙或排比句型。"
            advice = "建議將特徵詞替換為更自然的繁中詞彙（例如「凸顯出 ➜ 顯示」、「藍圖 ➜ 架構」）。"
        else:
            tier = "LOW"
            reason = "神經網路特徵高度符合天然人類寫作，用詞樸實精確，無機器套路痕跡。"
            advice = "保持原狀，此句為非常自然的人類原創範本。"

        all_sentences.append({
            "id": idx + 1,
            "text": s,
            "prob": calibrated_score,
            "tier": tier,
            "cliches": hit_words,
            "reason": reason,
            "advice": advice
        })

    sentence_scores = [r["prob"] for r in all_sentences]
    avg_s_prob = float(np.mean(sentence_scores))
    high_ratio = sum(1 for r in all_sentences if r["tier"] == "HIGH") / len(all_sentences)

    final_ai_prob = avg_s_prob * 0.80 + high_ratio * 0.20
    final_ai_prob = min(max(final_ai_prob, 0.02), 0.98)
    final_human_prob = 1.0 - final_ai_prob

    high_list = [r for r in all_sentences if r["tier"] == "HIGH"]
    med_list = [r for r in all_sentences if r["tier"] == "MED"]
    low_list = [r for r in all_sentences if r["tier"] == "LOW"]

    if final_ai_prob >= 0.60:
        verdict = "🚨 篇章高度疑似包含大量 AI 生成或深度改寫"
        verdict_color = "#be123c"
    elif final_ai_prob >= 0.22 or len(matched_cliches_all) >= 4:
        verdict = "⚠️ 偵測到局部 ChatGPT 繁中學術潤稿修飾 (AI-Polished)"
        verdict_color = "#c2410c"
    else:
        verdict = "✅ 傾向純人類獨立撰寫 (高原創性、低 AI 痕跡)"
        verdict_color = "#15803d"

    return {
        "final_ai_prob": final_ai_prob,
        "final_human_prob": final_human_prob,
        "verdict": verdict,
        "verdict_color": verdict_color,
        "total_chars": total_chinese_chars,
        "total_sentences": len(sentences),
        "high_count": len(high_list),
        "med_count": len(med_list),
        "low_count": len(low_list),
        "matched_cliches": list(set(matched_cliches_all)),
        "high_sentences": high_list,
        "med_sentences": med_list,
        "low_sentences": low_list,
        "all_sentences": all_sentences
    }

# 執行上傳
print("📂 請上傳繁體中文論文／文章檔案 (.docx, .pdf, .txt)：")
uploaded_docs = files.upload()

analysis_results = {}
for fname, content in uploaded_docs.items():
    with open(fname, "wb") as f:
        f.write(content)

    print(f"\n正在分析檔案: {fname}")
    raw_text = extract_text_from_file(fname)
    res = analyze_chinese_document_real(raw_text, model, tokenizer)
    if res:
        analysis_results[fname] = res

# 渲染診斷儀表板
if analysis_results:
    dashboard_template = """
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'PingFang TC', 'Microsoft JhengHei', sans-serif; background: #f8fafc; padding: 25px; border-radius: 12px;">
        <div style="max-width: 960px; margin: auto;">
            <div style="background: white; padding: 22px; border-radius: 10px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.08); margin-bottom: 20px;">
                <h2 style="color: #0f172a; margin: 0 0 8px 0;">🔬 繁體中文學術論文 AI 潤稿與原創性診斷系統 (嚴謹文獻裁切版)</h2>
                <p style="color: #64748b; font-size: 14px; margin: 0;"><b>[文獻區已精確過濾]</b> 自動排除參考文獻、目錄、標題。採用預訓練中文偵測模型進行逐句推論，完整呈現綠/黃/紅三類分佈。</p>
            </div>
            <div id="dash-root"></div>
        </div>
    </div>

    <style>
        .doc-card { background: white; border-radius: 10px; padding: 24px; margin-bottom: 24px; box-shadow: 0 2px 4px rgba(0,0,0,0.06); }
        .doc-title { font-size: 18px; font-weight: 700; color: #1e293b; margin-bottom: 15px; border-bottom: 2px solid #e2e8f0; padding-bottom: 10px; }
        .score-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 18px; }
        .score-box { padding: 16px; border-radius: 8px; text-align: center; }
        .ai-box { background: #fff1f2; border: 1px solid #fecdd3; }
        .human-box { background: #f0fdf4; border: 1px solid #bbf7d0; }
        .score-label { font-size: 13px; font-weight: 600; text-transform: uppercase; margin-bottom: 5px; }
        .score-val { font-size: 28px; font-weight: 800; }
        .progress-bar-bg { background: #e2e8f0; border-radius: 10px; height: 14px; overflow: hidden; display: flex; margin-bottom: 15px; }
        .progress-ai { background: #e11d48; height: 100%; }
        .progress-human { background: #16a34a; height: 100%; }
        .stat-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; background: #f8fafc; padding: 12px; border-radius: 6px; margin-bottom: 18px; font-size: 13px; color: #334155; }
        .cliche-tag { display: inline-block; background: #fee2e2; color: #991b1b; padding: 3px 8px; border-radius: 12px; font-size: 12px; margin: 3px; font-weight: 600; }

        .tab-container { display: flex; gap: 8px; border-bottom: 2px solid #e2e8f0; margin-bottom: 18px; padding-bottom: 4px; overflow-x: auto; }
        .tab-btn { background: #f1f5f9; border: none; outline: none; padding: 9px 16px; font-size: 13px; font-weight: 700; border-radius: 6px; cursor: pointer; color: #475569; transition: all 0.2s; }
        .tab-btn:hover { background: #e2e8f0; }
        .tab-btn.active { background: #0f172a; color: white; }
        .tab-pane { display: none; }
        .tab-pane.active { display: block; }

        .sentence-card { padding: 14px; margin-bottom: 12px; border-radius: 6px; }
        .sentence-card.high { background: #fff1f2; border-left: 5px solid #e11d48; }
        .sentence-card.med { background: #fffaf0; border-left: 5px solid #f97316; }
        .sentence-card.low { background: #f0fdf4; border-left: 5px solid #16a34a; }
        .sentence-header { display: flex; justify-content: space-between; font-size: 12px; font-weight: 700; margin-bottom: 6px; }
        .sentence-text { color: #1e293b; font-size: 14px; line-height: 1.7; margin-bottom: 8px; }
        .meta-box { background: rgba(255,255,255,0.7); padding: 8px 10px; border-radius: 4px; font-size: 12px; margin-top: 6px; line-height: 1.5; }
    </style>

    <script>
        (function() {
            const data = __DATA_JSON__;
            const root = document.getElementById('dash-root');

            function renderCardList(items, emptyMsg) {
                if (!items || items.length === 0) {
                    return `<div style="padding: 18px; background: #f8fafc; color: #64748b; border-radius: 6px; text-align: center; font-size: 14px;">${emptyMsg}</div>`;
                }
                return items.map(s => {
                    const sPct = (s.prob * 100).toFixed(1);
                    const tagInfo = s.cliches.length > 0 ? ` (命中 AI 套話: <b>${s.cliches.join('、')}</b>)` : '';
                    let badge = '';
                    let cClass = s.tier.toLowerCase();

                    if (s.tier === 'HIGH') {
                        badge = `<span style="color: #be123c;">🚨 高度疑似 AI 生成${tagInfo}</span>`;
                    } else if (s.tier === 'MED') {
                        badge = `<span style="color: #c2410c;">⚠️ 中度 AI 潤稿痕跡${tagInfo}</span>`;
                    } else {
                        badge = `<span style="color: #15803d;">🟢 較低 AI 痕跡句 (天然人類原創範本)</span>`;
                    }

                    return `
                        <div class="sentence-card ${cClass}">
                            <div class="sentence-header">
                                <span>#${s.id} ${badge}</span>
                                <span style="color: #64748b;">AI 判定值: <b>${sPct}%</b></span>
                            </div>
                            <div class="sentence-text">${s.text}</div>
                            <div class="meta-box">
                                <div><b>🔍 診斷分析：</b>${s.reason}</div>
                                <div style="color: #0369a1; margin-top: 3px;"><b>💡 繁中降重修訂指引：</b>${s.advice}</div>
                            </div>
                        </div>
                    `;
                }).join('');
            }

            Object.keys(data).forEach((fname, docIdx) => {
                const info = data[fname];
                const aiPct = (info.final_ai_prob * 100).toFixed(1);
                const humanPct = (info.final_human_prob * 100).toFixed(1);

                const card = document.createElement('div');
                card.className = 'doc-card';

                let clichesHtml = '';
                if (info.matched_cliches.length > 0) {
                    clichesHtml = info.matched_cliches.map(c => `<span class="cliche-tag">⚠️ ${c}</span>`).join(' ');
                } else {
                    clichesHtml = '<span style="color:#64748b; font-size:13px;">未發現明顯繁中典型 AI 轉折套話。</span>';
                }

                const uid = 'zh_doc_' + docIdx;

                card.innerHTML = `
                    <div class="doc-title">📄 繁中論文檔案: ${fname}</div>

                    <div class="score-grid">
                        <div class="score-box ai-box">
                            <div class="score-label" style="color: #be123c;">AI 痕跡 / 潤稿綜合指數</div>
                            <div class="score-val" style="color: #e11d48;">${aiPct}%</div>
                        </div>
                        <div class="score-box human-box">
                            <div class="score-label" style="color: #15803d;">人類原創獨立指數</div>
                            <div class="score-val" style="color: #16a34a;">${humanPct}%</div>
                        </div>
                    </div>

                    <div style="font-size: 13px; font-weight: 600; color: #475569; margin-bottom: 6px;">全文綜合比例：</div>
                    <div class="progress-bar-bg">
                        <div class="progress-ai" style="width: ${aiPct}%;"></div>
                        <div class="progress-human" style="width: ${humanPct}%;"></div>
                    </div>

                    <div class="stat-grid">
                        <div><b>診斷結果：</b> <span style="color:${info.verdict_color}; font-weight:bold;">${info.verdict}</span></div>
                        <div><b>有效論述中文字數：</b> ${info.total_chars} 字</div>
                        <div><b>分析有效總句數：</b> ${info.total_sentences} 句</div>
                        <div><b>分類分佈：</b> 🔴 ${info.high_count} | 🟡 ${info.med_count} | 🟢 ${info.low_count}</div>
                    </div>

                    <div style="margin-bottom: 18px;">
                        <div style="font-size: 13px; font-weight: 700; color: #334155; margin-bottom: 6px;">🔍 命中之 ChatGPT 典型繁中高頻特徵詞：</div>
                        ${clichesHtml}
                    </div>

                    <h4 style="color: #334155; margin: 22px 0 12px 0;">📑 全文句子分類對照清單（點選切換分類）：</h4>

                    <div class="tab-container">
                        <button class="tab-btn active" id="btn_high_${uid}">🚨 高度疑似 AI (${info.high_count} 句)</button>
                        <button class="tab-btn" id="btn_med_${uid}">⚠️ 中度 AI 潤稿 (${info.med_count} 句)</button>
                        <button class="tab-btn" id="btn_low_${uid}">🟢 較低 AI 痕跡 / 原創範本 (${info.low_count} 句)</button>
                        <button class="tab-btn" id="btn_all_${uid}">📜 全文依序對照 (${info.total_sentences} 句)</button>
                    </div>

                    <div class="tab-pane active" id="pane_high_${uid}">
                        ${renderCardList(info.high_sentences, '🎉 本篇未發現任何高度疑似 AI 生成的中文句子。')}
                    </div>
                    <div class="tab-pane" id="pane_med_${uid}">
                        ${renderCardList(info.med_sentences, '🎉 本篇未發現中度 AI 潤稿痕跡句子。')}
                    </div>
                    <div class="tab-pane" id="pane_low_${uid}">
                        ${renderCardList(info.low_sentences, '本篇較少純人類風格句子。')}
                    </div>
                    <div class="tab-pane" id="pane_all_${uid}">
                        ${renderCardList(info.all_sentences, '無句子資料。')}
                    </div>
                `;

                root.appendChild(card);

                const tabs = ['high', 'med', 'low', 'all'];
                tabs.forEach(t => {
                    const btn = document.getElementById(`btn_${t}_${uid}`);
                    btn.addEventListener('click', () => {
                        tabs.forEach(otherT => {
                            document.getElementById(`btn_${otherT}_${uid}`).classList.remove('active');
                            document.getElementById(`pane_${otherT}_${uid}`).classList.remove('active');
                        });
                        btn.classList.add('active');
                        document.getElementById(`pane_${t}_${uid}`).classList.add('active');
                    });
                });
            });
        })();
    </script>
    """
    rendered = dashboard_template.replace("__DATA_JSON__", json.dumps(analysis_results))
    display(HTML(rendered))

📂 請上傳繁體中文論文／文章檔案 (.docx, .pdf, .txt)：


Saving 20260416新聞初稿_徐臺屏老師.docx to 20260416新聞初稿_徐臺屏老師 (1).docx

正在分析檔案: 20260416新聞初稿_徐臺屏老師 (1).docx
ℹ️ 未偵測到獨立「參考文獻 / References」標題，進行全文掃描。
